# Long-run risks: the model and aggregate asset prices

Can a small, persistent component in consumption growth and slowly moving volatility explain the equity premium and the price-dividend ratio? [Bansal and Yaron (2004)](https://doi.org/10.1111/j.1540-6261.2004.00670.x) said yes, and the model became a standard one. [Beeler and Campbell (2012)](https://doi.org/10.1561/104.00000004) test it: they simulate the model at two calibrations, BY from Bansal and Yaron and BKY from Bansal, Kiku and Yaron, and ask what it implies for predictability, volatility, bond yields and the elasticity of intertemporal substitution. They find that it fits the basic moments and fails on much of the rest. All the model columns of Tables 2 to 8 are computed here from the parameters of Table 1 and printed beside the article's.

**Why this replication is partial.** A replication in Pollis normally recomputes everything from public data. This one cannot, and only the model side is recomputed. The stock returns, dividends and risk-free rates of the article come from CRSP, which is licensed, and the journal posts no data for this article. The consumption series come from the BEA and are public, but they are revised over time. So the data columns below are the values printed in the article, and the percentiles of the data in the simulated samples start from those printed values. The notebook downloads nothing. Section 14 goes through the sources one by one, with the public series nearest to each.

**What the published article left out.** Section 13 adds a real perpetuity. It was in the 2009 working paper, and the published article no longer has it.

The article, [from the journal's site](https://cfr.ivo-welch.org/published/papers/beeler2012longrun.pdf), and [the authors' appendix](https://campbell.scholars.harvard.edu/sites/g/files/omnuum5881/files/campbell/files/longrunrisks_appendix100611.pdf), which discusses the data sources and the modelling choices at length. The notebook checks its tables against the published article.

In [ ]:
Pollis.packages("Plots")

## Set-up

`using` for the plots and the standard library. `replicate` prints each value in full, rounded to the published precision, beside the published value, and how many decimals they share. `medians` and `percentiles`, defined after the simulation, call it for each published table.

In [ ]:
using Plots, Printf, Random, Statistics

"""
	replicate(title, rows)

Print each Julia value in full, rounded to the published precision and beside the published value, with the number of decimals they share.
Each row is `(label, value, published)`, with `published` written as in the source: its decimals give the published precision.
"""
function replicate(title, rows)
	@printf("\n%s\n%-26s%16s%14s%14s\n", title, "", "Julia", "rounded", "published")
	for (label, value, published) in rows
		p = parse(Float64, replace(published, "," => ""))
		d = occursin('.', published) ? length(split(published, '.')[end]) : 0  # decimals as published
		k = findlast(j -> round(value, digits=j) == round(p, digits=j), 0:d)  # position in 0:d
		@printf("%-26s%16.8f%14s%14s  %s\n", label, value, Printf.format(Printf.Format("%.$(d)f"), value), published, k === nothing ? "DIFFERENT" : "same to $(k - 1) decimals")
	end
end

## 1. The model

Monthly, with four independent standard normal shocks (§2, Table 1 on p. 148):

```
dc[t+1] = mu + x[t] + sqrt(v[t]) eta[t+1]
x[t+1]  = rho x[t] + phie sqrt(v[t]) e[t+1]
v[t+1]  = sigma^2 + nu (v[t] - sigma^2) + sw w[t+1]
dd[t+1] = mud + phi x[t] + varphi sqrt(v[t]) u[t+1] + pi_ sqrt(v[t]) eta[t+1]
```

The investor has Epstein-Zin preferences with risk aversion `gamma`, elasticity of intertemporal substitution `psi` and discount factor `delta`. Bansal and Yaron consider `gamma` of 7.5 and 10; the article uses 10 for both calibrations.

The two calibrations tell different stories, and the article's description of them (§2.3) can be checked by hand: the half-lives, the discount rate, and the average size of a monthly change in the expected long-run level of consumption and of dividends.

In [ ]:
"""
	LRR

The parameters of Table 1.
"""
Base.@kwdef struct LRR
	delta; mu; rho; phie; mud; phi; varphi; pi_; sigma; sw; nu; gamma = 10.0; psi = 1.5
end
variant(m; changes...) = LRR(; (f => getfield(m, f) for f in fieldnames(LRR))..., changes...)

const BY = LRR(delta = .998, mu = .0015, rho = .979, phie = .044, mud = .0015, phi = 3.0, varphi = 4.5, pi_ = 0.0, sigma = .0078, sw = .0000023, nu = .987)
const BKY = LRR(delta = .9989, mu = .0015, rho = .975, phie = .038, mud = .0015, phi = 2.5, varphi = 5.96, pi_ = 2.6, sigma = .0072, sw = .0000028, nu = .999)

halflife(a) = log(.5) / log(a)
longrun(m) = 100 * m.phie * m.sigma * sqrt(2 / pi) / (1 - m.rho)  # mean absolute monthly change, in percent
replicate("Section 2.3 (pp. 148-150)", [
	("half-life of x, BY, months", halflife(BY.rho), "33"),
	("half-life of x, BKY, months", halflife(BKY.rho), "27"),
	("half-life of v, BKY, years", halflife(BKY.nu) / 12, "58"),
	("time preference, BY, %", -1200 * log(BY.delta), "2.4"),
	("long-run c, BY, %", longrun(BY), "1.3"),
	("long-run c, BKY, %", longrun(BKY), "0.9"),
	("long-run d, BY, %", BY.phi * longrun(BY), "3.9"),
	("long-run d, BKY, %", BKY.phi * longrun(BKY), "2.2")])

## 2. The solution

The log wealth-consumption ratio and the log price-dividend ratio are linear in the states (the article's Appendix, after Campbell and Shiller):

```
z[t]  = A0 + A1 x[t] + A2 v[t]
zm[t] = A0m + A1m x[t] + A2m v[t]
```

The coefficients depend on the linearisation constants `k0`, `k1`, which depend on the mean of `z`, so each claim is a fixed point. Real zero-coupon bonds follow from the same stochastic discount factor, month by month: `p[n] = a[n] + b[n] x + c[n] v`. Without a fixed point the price of the claim is infinite, and `solve` returns `nothing`.

In [ ]:
kappas(z) = (k1 = exp(z) / (1 + exp(z)); (k0 = log(1 + exp(z)) - k1 * z, k1 = k1))

"""
	fixedpoint(f, z)

Iterate `z = f(z)` with damping; `nothing` if it does not settle at a finite value.
"""
function fixedpoint(f, z)
	for _ in 1:200_000
		new = f(z)
		(isfinite(new) && abs(new) < 1000) || return nothing
		abs(new - z) < 1e-10 && return z
		z = 0.5 * z + 0.5 * new
	end
	nothing
end

"""
	solve(m; months = 360)

The coefficients of the consumption claim, of the dividend claim and of the zero-coupon bonds up to `months`, with `a[n + 1]` for maturity `n`.
"""
function solve(m; months = 360)
	(; delta, mu, rho, phie, mud, phi, varphi, pi_, sigma, sw, nu, gamma, psi) = m
	th = (1 - gamma) / (1 - 1 / psi)
	function claim(z)
		(; k0, k1) = kappas(z)
		A1 = (1 - 1 / psi) / (1 - k1 * rho)
		A2 = 0.5 * th * ((1 - 1 / psi)^2 + (k1 * A1 * phie)^2) / (1 - k1 * nu)
		A0 = (log(delta) + (1 - 1 / psi) * mu + k0 + k1 * A2 * sigma^2 * (1 - nu) + 0.5 * th * k1^2 * A2^2 * sw^2) / (1 - k1)
		(; z = A0 + A2 * sigma^2, k0, k1, A0, A1, A2)
	end
	za = fixedpoint(z -> claim(z).z, 6.0)
	za === nothing && return nothing
	(; k0, k1, A0, A1, A2) = claim(za)
	const0 = th * log(delta) + (-th / psi + th - 1) * mu + (th - 1) * (k0 + k1 * A0 - A0 + k1 * A2 * sigma^2 * (1 - nu))
	function divclaim(z)
		km = kappas(z)
		A1m = (phi - 1 / psi) / (1 - km.k1 * rho)
		A2m = ((th - 1) * A2 * (k1 * nu - 1) + 0.5 * ((pi_ - gamma)^2 + varphi^2 + ((th - 1) * k1 * A1 * phie + km.k1 * A1m * phie)^2)) / (1 - km.k1 * nu)
		A0m = (const0 + mud + km.k0 + km.k1 * A2m * sigma^2 * (1 - nu) + 0.5 * sw^2 * ((th - 1) * k1 * A2 + km.k1 * A2m)^2) / (1 - km.k1)
		(; z = A0m + A2m * sigma^2, k0m = km.k0, k1m = km.k1, A0m, A1m, A2m)
	end
	zm = fixedpoint(z -> divclaim(z).z, 5.0)
	zm === nothing && return nothing
	a = zeros(months + 1); b = zeros(months + 1); c = zeros(months + 1)
	for n in 1:months
		vol = (th - 1) * k1 * A2 + c[n]
		a[n + 1] = a[n] + const0 + c[n] * sigma^2 * (1 - nu) + 0.5 * sw^2 * vol^2
		b[n + 1] = -1 / psi + rho * b[n]
		c[n + 1] = (th - 1) * A2 * (k1 * nu - 1) + nu * c[n] + 0.5 * (gamma^2 + ((th - 1) * k1 * A1 * phie + b[n] * phie)^2)
	end
	(; th, za, k0, k1, A0, A1, A2, const0, zm, divclaim(zm)..., a, b, c)
end

solutionBY, solutionBKY = solve(BY), solve(BKY)
for (name, s) in (("BY", solutionBY), ("BKY", solutionBKY))
	@printf("%-4s  mean z %.4f, k1 %.5f; mean zm %.4f, k1m %.5f\n", name, s.za, s.k1, s.zm, s.k1m)
end

## 3. The simulation

§2.4 (p. 150): monthly, 10 years of burn-in from the steady state, a negative variance replaced by a small positive number with the path kept, and 100,000 samples, whose medians are the model's values. One path serves every table: 948 months for the annual sample, 741 for the quarterly one, and 732 for the postwar variance ratios of Table 3. The article does not say how small its positive number is; here it is `1e-12`.

Yearly growth comes from the sum of the monthly levels, returns from the sum of the monthly log returns, and the log price-dividend ratio divides the last month's price by the dividends of the past twelve months. The first year only gives the base for the first growth rate.

Each block of 500 samples has its own seed, so the result does not depend on the number of threads. With `julia --threads=auto` the cell below takes a few minutes on a laptop; a smaller `nsim` is much quicker, and 5,000 samples already give most values to two decimals.

In [ ]:
const nsim = 100_000
const burn, months, offset = 120, 960, 12
const MATURITIES = [1, 3, 6, 12, 24, 60, 120, 240, 360]

"""
	path!(p, rng, m, s)

One monthly path: log levels `c` and `d`, returns `rm` and `rf`, the log price over monthly dividend `z`, the states `x` and `v` at the end of each month, and which months were censored.
"""
function path!(p, rng, m, s; floorv = 1e-12)
	(; mu, rho, phie, mud, phi, varphi, pi_, sigma, sw, nu) = m
	(; k0m, k1m, A0m, A1m, A2m, a, b, c) = s
	x = 0.0; v = sigma^2; lc = 0.0; ld = 0.0
	for t in 1:burn + months
		sd = sqrt(v)
		eta = randn(rng); e = randn(rng); u = randn(rng); w = randn(rng)
		dc = mu + x + sd * eta
		dd = mud + phi * x + varphi * sd * u + pi_ * sd * eta
		zm0 = A0m + A1m * x + A2m * v
		rf = -(a[2] + b[2] * x + c[2] * v)
		x = rho * x + phie * sd * e
		v = sigma^2 + nu * (v - sigma^2) + sw * w
		censored = v < 0
		censored && (v = floorv)
		zm = A0m + A1m * x + A2m * v
		lc += dc; ld += dd
		if t > burn
			i = t - burn
			p.c[i] = lc; p.d[i] = ld; p.rm[i] = k0m + k1m * zm - zm0 + dd; p.rf[i] = rf; p.z[i] = zm
			p.x[i] = x; p.v[i] = v; p.censored[i] = censored
		end
	end
	p
end

"""
	aggregate(p, k, span)

Periods of `k` months over `span` months: growth of the sums of levels, sums of returns, and the last month's price over the past twelve months' dividends.
"""
function aggregate(p, k, span)
	T = span ÷ k
	window(t) = offset + (t - 1) * k + 1:offset + t * k
	C = [sum(exp, view(p.c, window(t))) for t in 0:T]
	D = [sum(exp, view(p.d, window(t))) for t in 0:T]
	rm = [sum(view(p.rm, window(t))) for t in 1:T]
	rf = [sum(view(p.rf, window(t))) for t in 1:T]
	pd = [p.z[offset + t * k] + p.d[offset + t * k] - log(sum(exp, view(p.d, offset + t * k - 11:offset + t * k))) for t in 1:T]
	(; dc = diff(log.(C)), dd = diff(log.(D)), rm, rf, pd)
end

acf1(y) = (b = mean(y); sum((y[t] - b) * (y[t - 1] - b) for t in 2:length(y)) / sum(abs2, y .- b))
horizon(y, J) = [sum(view(y, t + 1:t + J)) for t in 1:length(y) - J]
r2(y, x) = cor(y, x)^2
ols_r2(y, X) = (e = y - X * (X \ y); 1 - sum(abs2, e) / sum(abs2, y .- mean(y)))
ar1residuals(y) = (X = hcat(ones(length(y) - 1), y[1:end - 1]); z = y[2:end]; vcat(0.0, z - X * (X \ z)))
varianceratio(y, K) = var([sum(view(y, t:t + K - 1)) for t in 1:length(y) - K + 1]) / (K * var(y))

# Two-stage least squares of y on r, the instruments being Z.
function tsls(y, r, Z)
	F = hcat(ones(size(Z, 1)), Z)
	X = hcat(ones(length(r)), F * (F \ r))
	(X \ y)[2]
end
# Table 8: psi from dc on r, and 1/psi from r on dc, at t, with r, dc and pd at t - 2 as instruments.
function eis(dc, r, pd)
	n = length(dc)
	Z = hcat(r[1:n - 2], dc[1:n - 2], pd[1:n - 2])
	(tsls(dc[3:n], r[3:n], Z), tsls(r[3:n], dc[3:n], Z))
end

"""
	leadlag!(add, name, dc, pd, Ks, js)

Figures 1 and 2: the slope and R2 of dc[t+j] + ... + dc[t+j+K-1] on pd[t], so that j = 1 is the regression of Table 4.
"""
function leadlag!(add, name, dc, pd, Ks, js)
	T = length(dc)
	sums = cumsum(vcat(0.0, dc))
	for K in Ks, j in js
		first_t, last_t = max(1, 1 - j), min(T, T - j - K + 1)
		xbar = mean(view(pd, first_t:last_t))
		y = [sums[t + j + K] - sums[t + j] for t in first_t:last_t]
		ybar = mean(y)
		sxy = sum((pd[t] - xbar) * (y[t - first_t + 1] - ybar) for t in first_t:last_t)
		sxx = sum(abs2(pd[t] - xbar) for t in first_t:last_t)
		add("$name b $K $j", sxy / sxx)
		add("$name r $K $j", sxy^2 / (sxx * sum(abs2, y .- ybar)))
	end
end

"""
	statistics(p, s; figures = true)

Every statistic of the tables, for one path, as parallel lists of labels and values.
"""
function statistics(p, s; figures = true)
	labels = String[]; values = Float64[]
	add(l, v) = (push!(labels, l); push!(values, v))
	Y = aggregate(p, 12, 948); Q = aggregate(p, 3, 741); Y61 = aggregate(p, 12, 732)
	# Table 2: yearly in percent; quarterly means times 4 and deviations times 2, the risk-free rate annualised first
	for (f, A, mscale, sscale, rfscale) in (("Y", Y, 100, 100, 100), ("Q", Q, 400, 200, 400))
		for (l, z) in (("dc", A.dc), ("dd", A.dd), ("rm", A.rm))
			add("T2$f E($l)", mean(z) * mscale); add("T2$f s($l)", std(z) * sscale); add("T2$f a($l)", acf1(z))
		end
		add("T2$f E(rf)", mean(A.rf) * rfscale); add("T2$f s(rf)", std(A.rf) * rfscale); add("T2$f a(rf)", acf1(A.rf))
		add("T2$f E(pd)", mean(A.pd)); add("T2$f s(pd)", std(A.pd)); add("T2$f a(pd)", acf1(A.pd))
	end
	# Appendix Table V and section 3.2: the premium is the mean log excess return plus half its variance
	ex = Y.rm - Y.rf
	add("premium", 100 * (mean(ex) + 0.5 * var(ex))); add("s(ex)", 100 * std(ex))
	# Section 3.1: four-year falls in consumption
	fall = [sum(view(Y.dc, t:t + 3)) for t in 1:length(Y.dc) - 3]
	add("worst fall", minimum(fall))
	add("fall and rise", any(t -> fall[t] < -.18 && fall[t + 4] > .19, 1:length(fall) - 4) ? 1.0 : 0.0)
	add("fall of 18.4% and rise", any(t -> fall[t] < -.184 && fall[t + 4] > .19, 1:length(fall) - 4) ? 1.0 : 0.0)
	# Table 3
	for (l, A) in (("1930", Y), ("1948", Y61)), (n, z) in (("c", A.dc), ("d", A.dd)), K in (2, 4, 6)
		add("T3 $n $l V$K", varianceratio(z, K))
	end
	# Tables 4 and 5
	for (f, A, Js) in (("Y", Y, (1, 3, 5)), ("Q", Q, (4, 12, 20)))
		e = A.rm - A.rf
		for (n, z) in (("ex", e), ("c", A.dc), ("d", A.dd)), J in Js
			add("T4$f $n $J", r2(horizon(z, J), A.pd[1:end - J]))
		end
		for (n, z) in (("ex", e), ("c", A.dc), ("d", A.dd))
			u = abs.(ar1residuals(z))
			for J in Js
				add("T5$f $n $J", r2(log.(horizon(u, J))[2:end], A.pd[2:end - J]))  # the first residual is not defined
			end
		end
	end
	# Table 6
	for (n, z, lag) in (("ex", ex, ex), ("c", Y.dc, Y.dc), ("d", Y.dd, Y.dd), ("rf", Y.rf, Y.dc)), J in (1, 3, 5)
		T = length(z) - J
		add("T6 $n $J", ols_r2(horizon(z, J), hcat(ones(T), lag[1:T], Y.pd[1:T], Y.rf[1:T])))
	end
	# Table 7
	x1 = reshape(view(p.x, offset + 1:months), 12, :); v1 = reshape(view(p.v, offset + 1:months), 12, :)
	x0 = reshape(view(p.x, offset:months - 1), 12, :); v0 = reshape(view(p.v, offset:months - 1), 12, :)
	sx1, sv1, sx0, sv0 = vec(sum(x1, dims = 1)), vec(sum(v1, dims = 1)), vec(sum(x0, dims = 1)), vec(sum(v0, dims = 1))
	xbar, vbar = mean(sx1) / 12, mean(sv1) / 12
	(; a, b, c) = s
	yieldof(n) = -(a[n + 1] + b[n + 1] * xbar + c[n + 1] * vbar) / n * 1200
	for n in MATURITIES
		add("T7 yield $n", yieldof(n))
	end
	for n in MATURITIES[2:end]
		add("T7 spread $n", yieldof(n) - yieldof(1))
		r = @. 12 * (a[n] - a[n + 1] + a[2]) + b[n] * sx1 + c[n] * sv1 + (b[2] - b[n + 1]) * sx0 + (c[2] - c[n + 1]) * sv0
		add("T7 excess $n", 100 * (mean(r) + 0.5 * var(r))); add("T7 vol $n", 100 * std(r))
	end
	# Table 8; the second panel gives the reciprocal of the coefficient of the asset on consumption growth
	for (f, A) in (("Y", Y), ("Q", Q)), (n, r) in (("rf", A.rf), ("rm", A.rm))
		psi, inverse = eis(A.dc, r, A.pd)
		add("T8$f $n psi", psi); add("T8$f $n inv", 1 / inverse)
	end
	# The censoring of section 2.4
	sample = view(p.censored, offset + 1:months)
	add("negative v", 100 * mean(sample)); add("path with negative v", 100 * any(sample))
	if figures
		leadlag!(add, "F1", Y.dc, Y.pd, (1, 3, 5), -4:4)
		leadlag!(add, "F2", Q.dc, Q.pd, (4, 12, 20), -15:15)
	end
	labels, values
end

"""
	simulate(m, n; figures = true)

Run `n` samples of the calibration `m`: a matrix with one row per statistic and one column per sample, with the row of each label.
"""
function simulate(m, n; figures = true, seed = 2012)
	s = solve(m)
	fresh() = (c = zeros(months), d = zeros(months), rm = zeros(months), rf = zeros(months), z = zeros(months), x = zeros(months), v = zeros(months), censored = falses(months))
	labels, _ = statistics(path!(fresh(), Xoshiro(1), m, s), s; figures)
	M = zeros(length(labels), n)
	Threads.@threads for block in collect(Iterators.partition(1:n, 500))
		rng = Xoshiro(seed + first(block))
		p = fresh()
		for k in block
			M[:, k] .= statistics(path!(p, rng, m, s), s; figures)[2]
		end
	end
	(; M, row = Dict(l => i for (i, l) in enumerate(labels)))
end
samples(sim, label) = view(sim.M, sim.row[label], :)
med(sim, label) = median(samples(sim, label))

seconds = @elapsed begin
	simBY = simulate(BY, nsim)
	simBKY = simulate(BKY, nsim)
end
@printf("%d samples of each calibration, %d threads, %.0f seconds\n", nsim, Threads.nthreads(), seconds)

const compared = NamedTuple[]
const shares = NamedTuple[]

"""
	medians(title, rows)

Print the model's medians beside the published ones, for both calibrations, and keep the comparison for section 12. Each row starts `(label, data, BY, BKY)`, all but the label as printed in the article.
"""
function medians(title, rows)
	for (name, sim, column) in (("BY", simBY, 3), ("BKY", simBKY, 4))
		shown = [(split(r[1], ' '; limit = 2)[2] * (isempty(r[2]) ? "" : ", data $(r[2])"), med(sim, r[1]), r[column]) for r in rows]
		replicate("$title, $name", shown)
		for (r, row) in zip(shown, rows)
			push!(compared, (; table = title, calibration = name, label = row[1], value = r[2], published = r[3]))
		end
	end
end

"""
	percentiles(title, rows)

The share of samples at or below the printed data value. That value is rounded, so the range in brackets takes it half a unit of its last digit either way. Each row is `(label, data, BY, BKY, percentile BY, percentile BKY)`.
"""
function percentiles(title, rows)
	@printf("\n%s, percentile of the data\n%-14s%8s%9s%18s%10s%9s%18s%10s\n", title, "", "data", "BY", "", "published", "BKY", "", "published")
	for (label, data, _, _, publishedBY, publishedBKY) in rows
		d = parse(Float64, data)
		half = 0.5 * 10.0^(-length(split(data, '.')[end]))
		@printf("%-14s%8s", split(label, ' '; limit = 2)[2], data)
		for (name, sim, published) in (("BY", simBY, publishedBY), ("BKY", simBKY, publishedBKY))
			z = samples(sim, label)
			low, high = mean(z .<= d - half), mean(z .<= d + half)
			@printf("%9.3f   [%5.3f, %5.3f]%10s", mean(z .<= d), low, high, published)
			push!(shares, (; table = title, calibration = name, label, low, high, published = parse(Float64, published)))
		end
		println()
	end
end
function both(title, rows)
	medians(title, rows)
	percentiles(title, rows)
end

replicate("Section 2.4, censored variance (p. 150), percent", [
	("months, BY (below 0.001)", mean(samples(simBY, "negative v")), "0.001"),
	("paths, BY (below 0.2)", mean(samples(simBY, "path with negative v")), "0.2"),
	("months, BKY", mean(samples(simBKY, "negative v")), "1.3")])
@printf("paths, BKY: %.2f percent (the article: \"over half\")\n", mean(samples(simBKY, "path with negative v")))

## 4. Table 2: the basic moments

`dc` and `dd` are the growth of consumption and dividends, `rm` the stock return, `rf` the risk-free rate, `pd` the log price-dividend ratio; `E`, `s` and `a` the mean, the standard deviation and the first-order autocorrelation. The autocorrelation is the sample one, with the whole sample's mean and variance, and this makes a difference of .01 here against a Pearson correlation of the series and its lag.

The model fits most of the means. It understates the volatility of the risk-free rate, and above all that of the price-dividend ratio: .18 against .45 in the yearly data (p. 154).

In [ ]:
medians("Table 2, yearly, 1930-2008 (p. 152)", [
	("T2Y E(dc)", "1.93", "1.80", "1.80"),
	("T2Y s(dc)", "2.16", "2.81", "2.50"),
	("T2Y a(dc)", "0.45", "0.47", "0.40"),
	("T2Y E(dd)", "1.15", "1.80", "1.81"),
	("T2Y s(dd)", "11.05", "11.20", "14.04"),
	("T2Y a(dd)", "0.21", "0.37", "0.26"),
	("T2Y E(rm)", "5.47", "6.75", "6.07"),
	("T2Y s(rm)", "20.17", "16.70", "18.68"),
	("T2Y a(rm)", "0.02", "0.02", "0.00"),
	("T2Y E(rf)", "0.56", "2.59", "1.23"),
	("T2Y s(rf)", "2.89", "1.21", "0.97"),
	("T2Y a(rf)", "0.65", "0.81", "0.80"),
	("T2Y E(pd)", "3.36", "3.01", "3.13"),
	("T2Y s(pd)", "0.45", "0.18", "0.18"),
	("T2Y a(pd)", "0.87", "0.66", "0.63")])
medians("Table 2, quarterly, 1947-2008 (p. 152)", [
	("T2Q E(dc)", "2.01", "1.80", "1.80"),
	("T2Q s(dc)", "1.02", "2.44", "2.26"),
	("T2Q a(dc)", "0.26", "0.31", "0.27"),
	("T2Q E(dd)", "2.29", "1.82", "1.80"),
	("T2Q s(dd)", "27.61", "10.53", "14.08"),
	("T2Q a(dd)", "-0.58", "0.26", "0.22"),
	("T2Q E(rm)", "6.36", "6.75", "6.02"),
	("T2Q s(rm)", "16.52", "16.54", "18.51"),
	("T2Q a(rm)", "0.08", "0.01", "0.00"),
	("T2Q E(rf)", "0.89", "2.59", "1.25"),
	("T2Q s(rf)", "1.82", "1.23", "0.97"),
	("T2Q a(rf)", "0.84", "0.94", "0.94"),
	("T2Q E(pd)", "3.46", "3.01", "3.14"),
	("T2Q s(pd)", "0.43", "0.18", "0.17"),
	("T2Q a(pd)", "0.98", "0.89", "0.86")])

## 5. The Great Depression, and what volatility adds

The article asks how likely 1930-1933 and 1934-1937 are in the model (§3.1, p. 151): a four-year fall in consumption of more than 18%, and such a fall followed by a four-year rise of 19%. It then removes the model's features one at a time (§3.2, p. 154): constant volatility (`sw = 0`), and growth with no predictable part as well (`phie = 0`).

The printed shares fit a fall of 18.4%, not of 18%: section 12 says more.

Volatility matters little in BY, and a good deal in BKY. These are smaller runs, a fifth of the samples, without the figures. The Appendix (Table V) also sets the elasticity of intertemporal substitution to 0.5. The BY calibration then has a finite price, the premium falls, and the risk-free rate rises to 6%; the BKY calibration's price is infinite, and the article prints NA.

In [ ]:
replicate("Section 3.1, percent of samples (p. 151)", [
	("fall of 18%, BY", 100 * mean(samples(simBY, "worst fall") .< -.18), "6.9"),
	("fall of 18%, BKY", 100 * mean(samples(simBKY, "worst fall") .< -.18), "6.6"),
	("then rise of 19%, BY", 100 * mean(samples(simBY, "fall and rise")), "0.1"),
	("then rise of 19%, BKY", 100 * mean(samples(simBKY, "fall and rise")), "0.3")])
@printf("\nThe fall that leaves the printed share: BY %.1f%%, BKY %.1f%%\n", -100 * quantile(samples(simBY, "worst fall"), .069), -100 * quantile(samples(simBKY, "worst fall"), .066))
@printf("With that fall, then a rise of 19%%: BY %.2f%%, BKY %.2f%%\n", 100 * mean(samples(simBY, "fall of 18.4% and rise")), 100 * mean(samples(simBKY, "fall of 18.4% and rise")))

smaller = nsim ÷ 5
function simpler(m, published)
	constant = simulate(variant(m; sw = 0.0), smaller; figures = false)
	iid = simulate(variant(m; sw = 0.0, phie = 0.0), smaller; figures = false)
	full = m === BY ? simBY : simBKY
	[(label, med(sim, statistic), value) for (label, sim, statistic, value) in published(iid, constant, full)]
end
replicate("Section 3.2, BY (p. 154)", simpler(BY, (iid, constant, full) -> [
	("premium, iid", iid, "premium", "0.0"), ("premium, constant vol", constant, "premium", "5.4"), ("premium, full", full, "premium", "5.5"),
	("s(pd), iid", iid, "T2Y s(pd)", "0.07"), ("s(pd), constant vol", constant, "T2Y s(pd)", "0.18"), ("s(pd), full", full, "T2Y s(pd)", "0.18")]))
replicate("Section 3.2, BKY (p. 154)", simpler(BKY, (iid, constant, full) -> [
	("premium, iid", iid, "premium", "1.6"), ("premium, constant vol", constant, "premium", "3.8"), ("premium, full", full, "premium", "6.6"),
	("s(pd), iid", iid, "T2Y s(pd)", "0.09"), ("s(pd), constant vol", constant, "T2Y s(pd)", "0.13"), ("s(pd), full", full, "T2Y s(pd)", "0.18")]))

lowEIS = simulate(variant(BY; psi = 0.5), smaller; figures = false)
statistic = ["T2Y E(rf)", "T2Y s(rf)", "premium", "s(ex)", "T2Y E(pd)", "T2Y s(pd)"]
replicate("Appendix Table V, BY, psi = 1.5", collect(zip(statistic, [med(simBY, l) for l in statistic], ["2.59", "1.21", "5.53", "16.42", "3.01", "0.18"])))
replicate("Appendix Table V, BKY, psi = 1.5", collect(zip(statistic, [med(simBKY, l) for l in statistic], ["1.23", "0.97", "6.57", "18.57", "3.13", "0.18"])))
replicate("Appendix Table V, BY, psi = 0.5", collect(zip(statistic, [med(lowEIS, l) for l in statistic], ["6.08", "3.59", "1.96", "12.95", "2.92", "0.11"])))
println("\nBKY with psi = 0.5, consumption claim: ", solve(variant(BKY; psi = 0.5)) === nothing ? "no fixed point, as the article's NA" : "a fixed point")

## 6. Table 3: variance ratios

The variance of K-year growth over K times the variance of one-year growth (p. 156). In the model it rises with K, since the persistent component adds up. In the long sample of data it falls: at six years the data are at the .001 and .007 percentiles for consumption. The postwar sample of Table 3 has 732 months, and yearly growth there too.

In [ ]:
both("Table 3, variance ratios (p. 156)", [
	("T3 c 1930 V2", "1.40", "1.47", "1.40", "0.271", "0.505"),
	("T3 c 1930 V4", "1.38", "2.01", "1.78", "0.039", "0.126"),
	("T3 c 1930 V6", "0.84", "2.32", "1.97", "0.001", "0.007"),
	("T3 d 1930 V2", "1.23", "1.37", "1.26", "0.111", "0.377"),
	("T3 d 1930 V4", "0.98", "1.71", "1.39", "0.008", "0.069"),
	("T3 d 1930 V6", "0.59", "1.87", "1.41", "0.000", "0.006"),
	("T3 c 1948 V2", "1.27", "1.46", "1.39", "0.087", "0.202"),
	("T3 c 1948 V4", "1.29", "1.96", "1.74", "0.047", "0.121"),
	("T3 c 1948 V6", "1.29", "2.22", "1.88", "0.065", "0.151"),
	("T3 d 1948 V2", "1.32", "1.36", "1.26", "0.388", "0.696"),
	("T3 d 1948 V4", "1.45", "1.66", "1.36", "0.290", "0.609"),
	("T3 d 1948 V6", "1.32", "1.79", "1.35", "0.191", "0.469")])

## 7. Table 4 and Figures 1 and 2: what the price-dividend ratio predicts

The R² of excess returns (`ex`), of consumption growth (`c`) and of dividend growth (`d`) over J years, or J quarters, on the log price-dividend ratio (p. 159). The data say returns are predictable and consumption is not. The model says the reverse, BY more than BKY.

The figures run the consumption regression at leads and lags of the ratio. The article draws three curves, and prints no values. Only the two model curves can be drawn here, since the third is the CRSP data. The vertical line, `j = 1`, is the regression of Table 4. Compare them by eye with pp. 162-163.

In [ ]:
both("Table 4, yearly (p. 159)", [
	("T4Y ex 1", "0.044", "0.007", "0.011", "0.918", "0.841"),
	("T4Y ex 3", "0.170", "0.017", "0.028", "0.980", "0.940"),
	("T4Y ex 5", "0.269", "0.025", "0.043", "0.990", "0.956"),
	("T4Y c 1", "0.060", "0.324", "0.145", "0.006", "0.202"),
	("T4Y c 3", "0.013", "0.350", "0.109", "0.002", "0.132"),
	("T4Y c 5", "0.000", "0.285", "0.085", "0.001", "0.015"),
	("T4Y d 1", "0.092", "0.404", "0.194", "0.001", "0.165"),
	("T4Y d 3", "0.059", "0.320", "0.084", "0.015", "0.399"),
	("T4Y d 5", "0.039", "0.255", "0.061", "0.039", "0.401")])
both("Table 4, quarterly (p. 159)", [
	("T4Q ex 4", "0.090", "0.008", "0.012", "0.980", "0.952"),
	("T4Q ex 12", "0.187", "0.022", "0.033", "0.970", "0.933"),
	("T4Q ex 20", "0.257", "0.033", "0.050", "0.969", "0.926"),
	("T4Q c 4", "0.000", "0.237", "0.063", "0.000", "0.023"),
	("T4Q c 12", "0.001", "0.269", "0.068", "0.003", "0.069"),
	("T4Q c 20", "0.002", "0.213", "0.060", "0.014", "0.089"),
	("T4Q d 4", "0.000", "0.159", "0.026", "0.001", "0.043"),
	("T4Q d 12", "0.001", "0.180", "0.029", "0.011", "0.115"),
	("T4Q d 20", "0.011", "0.147", "0.033", "0.084", "0.302")])

In [ ]:
function leadlagplot(figure, Ks, js, unit)
	panels = []
	for K in Ks, (statistic, ylabel) in (("b", "slope"), ("r", "R2"))
		curve(sim) = [med(sim, "$figure $statistic $K $j") for j in js]
		push!(panels, plot(js, curve(simBY); label = "BY", color = :black, ylabel, xlabel = "j, $unit", title = "K = $K", titlefontsize = 9))
		plot!(panels[end], js, curve(simBKY); label = "BKY", color = :black, linestyle = :dash)
		vline!(panels[end], [1]; label = "", color = :gray)
	end
	plot(panels...; layout = (length(Ks), 2), size = (720, 720), legend = :topleft, guidefontsize = 8, tickfontsize = 7, legendfontsize = 7)
end
leadlagplot("F1", (1, 3, 5), -4:4, "years")

In [ ]:
leadlagplot("F2", (4, 12, 20), -15:15, "quarters")

## 8. Table 5: what it says about volatility

Volatility here is the sum over J periods of the absolute residuals of an AR(1) of each series, and its log is regressed on the ratio (p. 165). In the data the ratio predicts the volatility of consumption, more in postwar data. BKY, built on persistent volatility, comes closer than BY but does not reach the postwar values.

In [ ]:
both("Table 5, yearly (p. 165)", [
	("T5Y ex 1", "0.001", "0.006", "0.026", "0.251", "0.118"),
	("T5Y ex 3", "0.001", "0.015", "0.082", "0.114", "0.042"),
	("T5Y ex 5", "0.001", "0.022", "0.107", "0.085", "0.032"),
	("T5Y c 1", "0.059", "0.006", "0.031", "0.974", "0.687"),
	("T5Y c 3", "0.148", "0.013", "0.103", "0.982", "0.623"),
	("T5Y c 5", "0.204", "0.020", "0.132", "0.980", "0.653"),
	("T5Y d 1", "0.032", "0.005", "0.034", "0.906", "0.482"),
	("T5Y d 3", "0.017", "0.013", "0.105", "0.554", "0.172"),
	("T5Y d 5", "0.007", "0.020", "0.133", "0.316", "0.096")])
both("Table 5, quarterly (p. 165)", [
	("T5Q ex 4", "0.002", "0.009", "0.106", "0.262", "0.052"),
	("T5Q ex 12", "0.033", "0.022", "0.177", "0.596", "0.159"),
	("T5Q ex 20", "0.061", "0.030", "0.183", "0.666", "0.238"),
	("T5Q c 4", "0.226", "0.008", "0.126", "1.000", "0.728"),
	("T5Q c 12", "0.437", "0.021", "0.197", "1.000", "0.891"),
	("T5Q c 20", "0.462", "0.030", "0.200", "0.999", "0.894"),
	("T5Q d 4", "0.049", "0.009", "0.128", "0.899", "0.274"),
	("T5Q d 12", "0.027", "0.021", "0.197", "0.557", "0.129"),
	("T5Q d 20", "0.019", "0.030", "0.199", "0.409", "0.109")])

## 9. Table 6: with the risk-free rate as well

The same predictions with more on the right: a constant, the lag of the variable, the ratio and the risk-free rate; for the risk-free rate itself, consumption growth, the ratio and the rate (p. 169). The model's two state variables should show up in these, and in the model they forecast consumption far better than they do in the data.

In [ ]:
both("Table 6 (p. 169)", [
	("T6 ex 1", "0.052", "0.033", "0.043", "0.704", "0.584"),
	("T6 ex 3", "0.171", "0.050", "0.086", "0.947", "0.812"),
	("T6 ex 5", "0.271", "0.061", "0.118", "0.973", "0.868"),
	("T6 c 1", "0.269", "0.405", "0.298", "0.125", "0.401"),
	("T6 c 3", "0.080", "0.392", "0.268", "0.010", "0.067"),
	("T6 c 5", "0.039", "0.321", "0.227", "0.010", "0.044"),
	("T6 d 1", "0.205", "0.467", "0.258", "0.004", "0.299"),
	("T6 d 3", "0.129", "0.341", "0.138", "0.036", "0.465"),
	("T6 d 5", "0.104", "0.280", "0.130", "0.072", "0.406"),
	("T6 rf 1", "0.483", "0.772", "0.698", "0.001", "0.027"),
	("T6 rf 3", "0.333", "0.539", "0.500", "0.056", "0.132"),
	("T6 rf 5", "0.325", "0.384", "0.379", "0.337", "0.372")])

## 10. Table 7: the real term structure

Real zero-coupon bonds (§5, p. 172): the mean yield, the spread over the one-month bond, the excess return (the yearly sum of monthly log excess returns, plus half its variance) and its volatility. The curve slopes down: long bonds hedge. The article compares this with TIPS and finds the model's long yields too low. There is no data column in the table.

The spread is the median of the sample's difference, not the difference of medians: the printed BKY spreads fit the first and not the second.

In [ ]:
medians("Table 7, maturity in months (p. 172)", [
	("T7 yield 1", "", "2.58", "1.22"),
	("T7 yield 3", "", "2.55", "1.19"),
	("T7 yield 6", "", "2.49", "1.15"),
	("T7 yield 12", "", "2.38", "1.07"),
	("T7 yield 24", "", "2.19", "0.91"),
	("T7 yield 60", "", "1.74", "0.51"),
	("T7 yield 120", "", "1.30", "-0.02"),
	("T7 yield 240", "", "0.91", "-0.88"),
	("T7 yield 360", "", "0.75", "-1.65"),
	("T7 spread 3", "", "-0.04", "-0.03"),
	("T7 spread 6", "", "-0.10", "-0.07"),
	("T7 spread 12", "", "-0.20", "-0.15"),
	("T7 spread 24", "", "-0.40", "-0.31"),
	("T7 spread 60", "", "-0.84", "-0.70"),
	("T7 spread 120", "", "-1.28", "-1.22"),
	("T7 spread 240", "", "-1.68", "-2.09"),
	("T7 spread 360", "", "-1.84", "-2.85"),
	("T7 excess 3", "", "-0.08", "-0.06"),
	("T7 excess 6", "", "-0.19", "-0.14"),
	("T7 excess 12", "", "-0.39", "-0.29"),
	("T7 excess 24", "", "-0.74", "-0.57"),
	("T7 excess 60", "", "-1.39", "-1.24"),
	("T7 excess 120", "", "-1.84", "-2.09"),
	("T7 excess 240", "", "-2.05", "-3.47"),
	("T7 excess 360", "", "-2.08", "-4.68"),
	("T7 vol 3", "", "0.16", "0.13"),
	("T7 vol 6", "", "0.38", "0.32"),
	("T7 vol 12", "", "0.79", "0.66"),
	("T7 vol 24", "", "1.48", "1.21"),
	("T7 vol 60", "", "2.80", "2.28"),
	("T7 vol 120", "", "3.77", "3.37"),
	("T7 vol 240", "", "4.31", "5.40"),
	("T7 vol 360", "", "4.41", "7.40")])

## 11. Table 8: the elasticity of intertemporal substitution

In the model it is 1.5. Two-stage least squares, with the asset's return, consumption growth and the ratio at `t - 2` as instruments (p. 176). `psi` is the coefficient of consumption growth on the asset's return. `inv` is the reciprocal of the coefficient of the return on consumption growth, as the second panel prints it. On the risk-free rate the model gets back close to 1.5; on stocks, in the model as in the data, the estimate is near zero though the true value is 1.5.

In [ ]:
both("Table 8, yearly (p. 176)", [
	("T8Y rf psi", "-0.123", "1.654", "1.324", "0.000", "0.020"),
	("T8Y rm psi", "-0.041", "0.041", "0.018", "0.395", "0.310"),
	("T8Y rf inv", "-0.679", "1.816", "1.871", "0.001", "0.025"),
	("T8Y rm inv", "-0.137", "0.428", "0.160", "0.445", "0.399")])
both("Table 8, quarterly (p. 176)", [
	("T8Q rf psi", "0.306", "1.495", "1.343", "0.001", "0.017"),
	("T8Q rm psi", "0.004", "0.030", "0.008", "0.471", "0.484"),
	("T8Q rf inv", "0.504", "1.613", "1.694", "0.000", "0.007"),
	("T8Q rm inv", "1.970", "0.428", "0.115", "0.845", "0.928")])

## 12. What differs, and why

The medians first, by how far each is from the printed value in units of its last digit. Then how far a gap could come from chance. The article's 100,000 samples and ours are two independent runs, so the difference of two medians has a standard deviation of √2 times each one's standard error, here from blocks of 1,000 samples, and rounding the printed value adds up to half a unit.

In [ ]:
function standarderror(sim, label; block = 1000)
	z = samples(sim, label)
	std([median(view(z, r)) for r in Iterators.partition(1:length(z), block)]) / sqrt(length(z) ÷ block)
end
decimals(s) = occursin('.', s) ? length(split(s, '.')[end]) : 0
units(r) = abs(round(r.value, digits = decimals(r.published)) - parse(Float64, r.published)) * 10.0^decimals(r.published)
gaps = round.(Int, units.(compared))
println(length(compared), " medians compared. Off by, in units of the last printed digit:")
for k in 0:3
	@printf("  %d: %d\n", k, count(==(k), gaps))
end
@printf("  4 or more: %d\n", count(>=(4), gaps))
function chance(r)
	se = standarderror(r.calibration == "BY" ? simBY : simBKY, r.label)
	half = 0.5 * 10.0^(-decimals(r.published))
	max(abs(r.value - parse(Float64, r.published)) - half, 0) / (sqrt(2) * se)
end
@printf("\n%-34s%-5s%-14s%10s%11s%20s\n", "Off by two or more units", "", "", "Julia", "published", "gap / (sqrt(2) se)")
for r in compared[gaps .>= 2]
	@printf("%-34s%-5s%-14s%10.3f%11s%20.1f\n", r.table[1:min(end, 32)], r.calibration, split(r.label, ' '; limit = 2)[2], r.value, r.published, chance(r))
end
z = [chance(r) for r in compared]
@printf("\nGap beyond rounding, of all the medians: over 2 standard deviations %d, over 3 %d\n", count(>(2), z), count(>(3), z))
function chanceshare(r)
	q = clamp((r.low + r.high) / 2, 1 / nsim, 1 - 1 / nsim)
	max(r.low - r.published - .0005, r.published - .0005 - r.high, 0) / sqrt(2 * q * (1 - q) / nsim)
end
w = chanceshare.(shares)
println("\n", length(shares), " printed percentiles. Beyond the range from rounding both values: ", count(>(0), w), ", over 2 standard deviations ", count(>(2), w), ", over 3 ", count(>(3), w))
for (r, g) in zip(shares, w)
	g > 2 && @printf("  %-32s%-5s%-14s[%.3f, %.3f], published %.3f, %.1f\n", r.table[1:min(end, 30)], r.calibration, split(r.label, ' '; limit = 2)[2], r.low, r.high, r.published, g)
end

**What this run gives** (100,000 samples, as in the article; the seeds are fixed, so the values above come back the same).

The medians: 262 compared. 178 agree to every printed digit, 62 are off by one in the last digit, 12 by two, 3 by three and 7 by four to six. The widest gaps are in Table 8, in the reciprocal coefficient on stock returns (0.422 against 0.428, BY, yearly), whose estimate is very noisy, and in the volatility of stock returns in Table 2 (18.72 against 18.68, BKY, yearly; 18.48 against 18.51, BKY, quarterly). None of the 262 is more than two standard deviations from the printed value, after rounding: what is left is what two independent simulations give.

The percentiles: 136 compared, each from a data value printed to two or three decimals. For 105 of them the printed percentile lies in the range that rounding allows. 31 lie outside it, 28 of them by less than two standard deviations. Three lie further: 0.988 against 0.990 (Table 4, BY, excess returns over 5 years), 0.392 to 0.395 against 0.401 and 0.126 to 0.128 against 0.132 (Table 6, BKY, consumption over 1 year and the risk-free rate over 3 years). With 136 comparisons, about six beyond two standard deviations would be expected by chance. The first of the three is at 3.1, and it is not settled here.

The Great Depression shares of section 5 are the one place where the literal text does not give the printed value: with a fall of 18% the shares are 7.8% and 7.3%, printed as 6.9% and 6.6%. A fall of 18.4% gives both of the printed shares, and with it the fall and rise comes to 0.06% and 0.34%, printed as 0.1% and 0.3%. So the article's "18%" looks like a rounded 18.4%. Without its series this cannot be checked here, and it is one number fitted to two shares.

**Choices the article leaves open, taken because they fit what it prints.** The first-order autocorrelation is the sample one, with the whole sample's mean and variance; a Pearson correlation gives .01 more. In Table 7 the spread is the median of each sample's difference. In the second panel of Table 8 the column is the reciprocal of the coefficient. In the figures, the sum runs from `t + j` to `t + j + K - 1`: the equation printed over Figure 1 ends at `t + j + K`, and the sum here puts the peaks where the article draws them. The small positive number that replaces a negative variance is `1e-12`, since the article gives none.

## 13. A real perpetuity, from the 2009 working paper

The perpetuity is not in the published article. The working-paper version, [NBER Working Paper 14788](https://www.nber.org/papers/w14788), has it in Table IX, Panel A for BY and Panel B for BKY: the mean yield of a consol paying one unit of consumption in every future period, its spread over the risk-free rate, the term premium `E(r_b - r^f) + var(r_b - r^f) / 2` and the standard deviation of the excess return, from 1.2 million simulated months, with the monthly excess returns summed to years. The version of October 2011, the published article and its appendix do not mention perpetuities or consols, and do not say why the table went. The article's method carries over: the same fixed point, for a claim with no growth, the same log-linear return, and a negative variance replaced by a small positive number. That replacement matters in BKY, whose variance is so persistent: on this path it happens in about 1% of months, and the working paper reports 1.3%. Where the working paper prints NA the price is infinite, and there is no fixed point.

In [ ]:
function consolmap(m, s, zl)
	(; delta, mu, rho, phie, sigma, nu, sw, gamma, psi) = m
	(; k0, k1, A0, A1, A2, th) = s
	kl1, kl0 = kappas(zl).k1, kappas(zl).k0
	Al1 = -(1 / psi) / (1 - kl1 * rho)
	Al2 = ((th - 1) * A2 * (k1 * nu - 1) + 0.5 * (gamma^2 + ((th - 1) * k1 * A1 * phie + kl1 * Al1 * phie)^2)) / (1 - kl1 * nu)
	c = th * log(delta) + (-th / psi + th - 1) * mu + (th - 1) * (k0 + k1 * A0 - A0 + k1 * A2 * sigma^2 * (1 - nu)) + kl0 + kl1 * Al2 * sigma^2 * (1 - nu) + 0.5 * sw^2 * ((th - 1) * k1 * A2 + kl1 * Al2)^2
	Al0 = c / (1 - kl1)
	(; z = Al0 + Al2 * sigma^2, Al0, Al1, Al2)
end

"""
	consolroot(m, s)

The first root of `z = consolmap(z)` on a wide scan, or `nothing`.
"""
function consolroot(m, s; grid = 0.0:0.002:40.0)
	g(z) = consolmap(m, s, z).z - z
	for (u, w) in zip(grid[1:end - 1], grid[2:end])
		if g(u) * g(w) <= 0
			for _ in 1:80
				h = (u + w) / 2
				g(h) * g(u) <= 0 ? (w = h) : (u = h)
			end
			return (u + w) / 2
		end
	end
	nothing
end

"""
	states(m; n = 1_200_000, seed = 1)

A long path of `x` and `v`, with a negative `v` replaced by a small positive number (§2.4), for the unconditional means.
"""
function states(m; n = 1_200_000, seed = 1, sqrtshock = false)
	rng = Xoshiro(seed); x = 0.0; v = m.sigma^2
	xs = zeros(n); vs = zeros(n)
	for t in 1:n
		xs[t] = x; vs[t] = v
		s = sqrt(v)
		x = m.rho * x + m.phie * s * randn(rng)
		v = m.sigma^2 + m.nu * (v - m.sigma^2) + m.sw * (sqrtshock ? s / m.sigma : 1.0) * randn(rng)
		v < 0 && (v = sqrtshock ? 0.0 : 1e-12)
	end
	xs, vs
end

"""
	consolmoments(m)

The four moments of Table IX on one path, with their standard errors from 100 blocks of it.
"""
function consolmoments(m)
	s = solve(m)
	root = consolroot(m, s)
	b = consolmap(m, s, root)
	k = kappas(root)
	xs, vs = states(m)
	z = @. b.Al0 + b.Al1 * xs + b.Al2 * vs
	rf = @. -(s.a[2] + s.b[2] * xs + s.c[2] * vs)
	yields = @. 1200 * exp(-z)
	spread = yields .- 1200 .* rf
	excess = [sum(k.k0 + k.k1 * z[t + 1] - z[t] - rf[t] for t in u:u + 11) for u in 1:12:length(z) - 12]
	premium(e) = 100 * (mean(e) + var(e) / 2)
	sd(e) = 100 * std(e)
	se(y, f) = std([f(view(y, r)) for r in Iterators.take(Iterators.partition(1:length(y), length(y) ÷ 100), 100)]) / 10
	(; yield = mean(yields), spread = mean(spread), premium = premium(excess), sd = sd(excess), se = (se(yields, mean), se(spread, mean), se(excess, premium), se(excess, sd)), replaced = 100 * mean(<=(1e-12), vs))
end

for (panel, base, gamma, psi, printed) in (
	("A", BY, 5.0, 0.5, ("3.98", "-1.95", "-1.50", "9.85")), ("A", BY, 5.0, 1.5, ("2.25", "-0.86", "-0.80", "3.58")), ("A", BY, 5.0, 2.0, ("2.05", "-0.69", "-0.65", "2.76")),
	("A", BY, 10.0, 0.5, ("0.45", "-5.66", "-4.97", "11.82")), ("A", BY, 10.0, 1.5, ("0.45", "-2.14", "-2.05", "4.37")), ("A", BY, 10.0, 2.0, ("0.45", "-1.63", "-1.57", "3.45")),
	("B", BKY, 5.0, 0.5, ("3.05", "-1.49", "-1.16", "9.47")))
	c = consolmoments(variant(base; gamma, psi))
	replicate("Working paper Table IX, Panel $panel, gamma = $gamma, psi = $psi", [("E(y_b)", c.yield, printed[1]), ("E(y_b - r^f)", c.spread, printed[2]), ("E(r_b - r^f) + var / 2", c.premium, printed[3]), ("sd(r_b - r^f)", c.sd, printed[4])])
	@printf("standard errors, from 100 blocks of the path: yield %.3f, spread %.3f, premium %.3f, sd %.3f; months with v replaced: %.3f%%\n", c.se..., c.replaced)
end

"""
	consolprice(m)

Whether the consumption claim and the consol have finite prices.
"""
function consolprice(m)
	s = solve(m)
	s === nothing ? "no claim" : consolroot(m, s) === nothing ? "no consol" : "finite"
end

println("\nFinite prices, psi = 0.5, 1.5, 2.0 (the NA of Table IX)")
for (name, base) in (("BY", BY), ("BKY", BKY)), gamma in (5.0, 10.0, 15.0)
	@printf("%-4s gamma = %-5s%s\n", name, gamma, join((rpad(consolprice(variant(base; gamma, psi)), 11) for psi in (0.5, 1.5, 2.0))))
end

println("\nThe lower discount factors of the notes to Table IX")
for (name, base, gamma, deltas) in (("BY", BY, 15.0, (.995, .996, .996)), ("BKY", BKY, 5.0, (.9989, .998, .998)), ("BKY", BKY, 10.0, (.996, .996, .996)), ("BKY", BKY, 15.0, (.993, .993, .993)))
	@printf("%-4s gamma = %-5s%s\n", name, gamma, join((rpad(string(delta, " ", consolprice(variant(base; gamma, psi, delta))), 18) for (psi, delta) in zip((0.5, 1.5, 2.0), deltas))))
end

The repeated 0.45 at `gamma = 10` looks like a misprint and is not one: the yields at the three values of `psi` come out the same to two decimals (0.451, 0.448, 0.448). Of Panel A's 24 values, 9 agree to every printed digit, 6 differ by .01, and 9 by .02 to .05. All are within about two standard deviations of the difference between two draws of 1.2 million months.

Panel B has one finite cell. Its spread, premium and standard deviation are within 1.5 such standard deviations. Its yield is not: 3.22 against 3.05, about 6.5 of them. This is not explained. Other treatments of a negative variance do not resolve it: kept negative it gives 3.63, on three paths, and replaced only where it is used 3.44, on six. Linearising the consol at its simulated mean gives 3.14, also on six, but moves the standard deviation further away, to 9.71. These were tried outside this notebook.

Every NA of both panels is reproduced, with the reason the notes to the table give, and every lower discount factor in those notes gives a finite price.

**Added in Pollis, not in the article or the working paper.** The consol's log-linear yield has an exact counterpart in the same model: the sum of the prices of all the zero-coupon bonds. That leaves the log-linearisation of the consol out, while the consumption claim stays log-linear. The panel *Continuous-Time Macro-Finance*, in its notebook tutorial on long-run risks, solves the model by a PDE, whose variance has a shock scaled by `sqrt(v)` where BY's is Gaussian, so the third column makes that one change in the discrete model. At `gamma = 10` that change does most of the work, from 0.445 to 0.391. That tutorial carries the steps on: the package's annual parameters in the continuous limit 0.386, then the PDE with the exact wealth ratio 0.365. At `gamma = 5` the same steps stay close: 2.250, 2.249, 2.244, 2.217.

In [ ]:
# Added in Pollis, not in the article or the working paper

"""
	claimsqrt(m)

The consumption claim with a shock to `v` scaled by `sqrt(v) / sigma`: the smaller root of the quadratic in `A2`.
"""
function claimsqrt(m)
	(; delta, mu, rho, phie, sigma, nu, sw, gamma, psi) = m
	th = (1 - gamma) / (1 - 1 / psi)
	function claim(z)
		(; k0, k1) = kappas(z)
		A1 = (1 - 1 / psi) / (1 - k1 * rho)
		q = 0.5 * th * ((1 - 1 / psi)^2 + (k1 * A1 * phie)^2)
		aa = 0.5 * th * k1^2 * sw^2 / sigma^2; bb = -(1 - k1 * nu)
		A2 = (-bb - sqrt(bb^2 - 4 * aa * q) * sign(-bb)) / (2 * aa)
		A0 = (log(delta) + (1 - 1 / psi) * mu + k0 + k1 * A2 * sigma^2 * (1 - nu)) / (1 - k1)
		(; z = A0 + A2 * sigma^2, k0, k1, A0, A1, A2)
	end
	(; claim(fixedpoint(z -> claim(z).z, 6.0))..., th)
end

"""
	sumofzeros(m, a, xs, vs; sqrtshock = false)

The mean yield of a consol as the sum of the zero-coupon prices, in percent, on every 100th state of the path.
"""
function sumofzeros(m, a, xs, vs; sqrtshock = false, maturities = 400_000)
	(; delta, mu, rho, phie, sigma, nu, sw, gamma, psi) = m
	(; k0, k1, A0, A1, A2, th) = a
	onv = sqrtshock ? sw^2 / sigma^2 : 0.0; constant = sqrtshock ? 0.0 : sw^2
	const0 = th * log(delta) + (-th / psi + th - 1) * mu + (th - 1) * (k0 + k1 * A0 - A0 + k1 * A2 * sigma^2 * (1 - nu))
	x, v = xs[1:100:end], vs[1:100:end]
	P = zeros(length(x))
	an, bn, cn = 0.0, 0.0, 0.0
	for n in 1:maturities
		vol = (th - 1) * k1 * A2 + cn
		an, bn, cn = an + const0 + cn * sigma^2 * (1 - nu) + 0.5 * constant * vol^2, -1 / psi + rho * bn,
			(th - 1) * A2 * (k1 * nu - 1) + nu * cn + 0.5 * (gamma^2 + ((th - 1) * k1 * A1 * phie + bn * phie)^2) + 0.5 * onv * vol^2
		if n <= 6000 || n % 50 == 0
			w = n <= 6000 ? 1.0 : 50.0
			@. P += w * exp(an + bn * x + cn * v)
		end
		n > 6000 && exp(an + bn * mean(x) + cn * mean(v)) * 50 < 1e-9 * mean(P) && break
	end
	1200 * mean(1 ./ P)
end

@printf("\n%-10s%22s%24s%22s\n", "psi = 1.5", "log-linear consol", "sum of zeros, Gaussian", "sum of zeros, sqrt(v)")
for gamma in (5.0, 10.0)
	m = variant(BY; gamma)
	s = solve(m)
	b = consolmap(m, s, consolroot(m, s))
	xs, vs = states(m)
	xq, vq = states(m; sqrtshock = true)
	@printf("gamma %-4s%22.3f%24.3f%22.3f\n", gamma, mean(@. 1200 * exp(-(b.Al0 + b.Al1 * xs + b.Al2 * vs))), sumofzeros(m, s, xs, vs), sumofzeros(m, claimsqrt(m), xq, vq; sqrtshock = true))
end

## 14. The article's data, source by source

The article's §3.1 and the Appendix's Section 2 describe the sources. Annual samples run from 1930 to 2008, quarterly ones from 1947.2 to 2008.4.

| Series | Source in the article | Can it be obtained? | Nearest public series |
|---|---|---|---|
| Consumption: real nondurables and services per capita | BEA, Table 7.1 | Yes, but revised: not the version the authors used | [BEA's NIPA files](https://apps.bea.gov/national/Release/TXT/NipaDataA.txt), series A796RX and A797RX, annual from 1929, quarterly from 1947Q1 |
| Stock returns, with and without dividends | CRSP, VWRETD and VWRETX | No, licensed | [Ken French's data library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html), made from CRSP, monthly from July 1926: the market (Mkt-RF + RF), and ten size portfolios with and without dividends |
| Dividends and the price-dividend ratio | Built from VWRETD and VWRETX | No, licensed | The same ten portfolios, weighted by number of firms times average size. [Shiller's data](https://shillerdata.com) have the S&P index, a different index |
| Inflation | BLS, CPI | Yes | FRED: [CPIAUCNS](https://fred.stlouisfed.org/series/CPIAUCNS), not seasonally adjusted, from 1913; [CPIAUCSL](https://fred.stlouisfed.org/series/CPIAUCSL), seasonally adjusted, from 1947 |
| Risk-free rate: three-month yield | CRSP, Fama Risk Free Rates | No, licensed | FRED: [TB3MS](https://fred.stlouisfed.org/series/TB3MS), on a discount basis, from January 1934, so without 1930-1933. French's RF is a one-month rate, from July 1926 |
| Ex ante real rate | The authors' regression on that yield and past inflation | Only with the proxies above | Not tried |

The article's own files cannot be downloaded. The journal's [list of articles](https://cfr.ivo-welch.org/published/) gives this one a PDF, an abstract and a BibTeX entry, and its [page at Emerald](https://www.emerald.com/cfr/issue/1/1) a PDF, with no Supplementary data, which recent articles there have. The journal [asks authors](https://cfr.ivo-welch.org/) to make programs or data available to data-licensed researchers on request.

How near the proxies come, yearly, 1930-2008, computed once on 1 November 2026 outside this notebook, with the CPI column of Shiller's file, December to December. Each cell gives the mean, the standard deviation and the first-order autocorrelation:

| Series | Printed in the article | From public data | Public data used |
|---|---|---|---|
| Consumption growth | 1.93, 2.16, .45 | 1.96, 2.14, .49 | BEA |
| Dividend growth | 1.15, 11.05, .21 | 1.17, 10.97, .16 | French's ten portfolios |
| Stock return | 5.47, 20.17, .02 | 5.53, 20.15, .02 | French's market |
| Risk-free rate | .56, 2.89, .65 | .47, 3.94, .63 | French's RF, ex post, so not the same quantity |
| Price-dividend ratio | 3.36, .45, .87 | 3.38, .48, .90 | French's ten portfolios |

Quarterly consumption growth from the BEA gives 2.06, 1.00, .25 against the printed 2.01, 1.02, .26. The other quarterly columns were not computed.

The proxies come close, but they are not the article's data, so the notebook keeps the printed values.¹

¹ With a CRSP licence, the data side of Tables 2 to 6 and Table 8 could be recomputed from the sources in the Appendix, whose Section 2 describes them. The risk-free rate in the data is an ex ante one, from a regression, which would have to be redone too.

## References

- Bansal, R., Kiku, D. and Yaron, A. (2012). An empirical evaluation of the long-run risks model for asset prices. *Critical Finance Review* 1(1), 183-221. doi:10.1561/104.00000005
- Bansal, R. and Yaron, A. (2004). Risks for the long run: a potential resolution of asset pricing puzzles. *The Journal of Finance* 59(4), 1481-1509. doi:10.1111/j.1540-6261.2004.00670.x
- Beeler, J. and Campbell, J. Y. (2012). The long-run risks model and aggregate asset prices: an empirical assessment. *Critical Finance Review* 1(1), 141-182. doi:10.1561/104.00000004
- Beeler, J. and Campbell, J. Y. (2009). The long-run risks model and aggregate asset prices: an empirical assessment. NBER Working Paper 14788. [nber.org/papers/w14788](https://www.nber.org/papers/w14788)
- The article, [from the journal's site](https://cfr.ivo-welch.org/published/papers/beeler2012longrun.pdf), and [the authors' appendix](https://campbell.scholars.harvard.edu/sites/g/files/omnuum5881/files/campbell/files/longrunrisks_appendix100611.pdf).
- Data: the data columns are the values printed in the article, from CRSP (licensed), the BEA and the BLS. No data is downloaded. Section 14 gives the public series nearest to each.

The article's first page prints the DOI 10.1561/103.00000004. Crossref gives that DOI to another article (*Data Envelopment Analysis Journal* 1(1), 29-55), and 10.1561/104.00000004 to this one, which is the DOI used here.